# Learning Buddy - uji pencarian konteks (buku Kurikulum Merdeka)

Menguji layanan pencarian persis seperti website: potongan **Recursive**, penyandi
Qwen3-Embedding-4B + FAISS, cross-encoder bge-reranker-v2-m3, dan **3 potongan** yang
dikirim ke Act Agent. Soal uji: 1.014 soal `Validasi_Soal_PJOK.xlsx`.

**Cara menjalankan**

1. Input: Dataset **learning-buddy-kode** (versi terbaru dari `python deploy/kemas.py`).
2. *Settings*: Accelerator **GPU T4 x2**, Internet **On**.
3. **Save Version > Save & Run All (Commit)**, sekitar 30-40 menit. Browser boleh ditutup.
4. Unduh tab *Output*: `hasil/Evaluasi_Pencarian.xlsx`, `hasil/evaluasi_pencarian.json`,
   `hasil/ambang_relevan.json`. Output notebook ini juga bisa ditambahkan sebagai input
   notebook server agar ambang pengecekan 1 tidak perlu dihitung ulang.

In [ ]:
# Salin kode ke folder kerja
import glob, os, shutil, subprocess, sys, time, re, json

# Matikan sisa server/tunnel dari jalankan sebelumnya. Proses latar itu TIDAK ikut mati saat
# kernel di-restart dan tetap memegang memori GPU (penyebab CUDA out of memory).
for nama in ('tb.server', 'cloudflared'):
    subprocess.run(['pkill', '-f', nama], check=False)
time.sleep(3)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
# indeks FAISS disimpan di luar folder kerja agar tidak dibangun ulang setiap Run All
os.environ.setdefault('TB_FAISS', '/kaggle/working/faiss_cache')
print(subprocess.run(['nvidia-smi', '--query-gpu=index,memory.used,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout)

# dicari di mana pun dalam input: zip, folder LearningBuddy, atau isi foldernya langsung
hit = glob.glob('/kaggle/input/**/tb/agen.py', recursive=True)
assert hit, 'Dataset learning-buddy-kode belum ditambahkan sebagai input.'
# Bila ada lebih dari satu salinan kode (dataset lama ikut terpasang), pakai yang terbaru:
# yang sudah punya tb/validasi_recursive.py, lalu yang berkasnya paling baru.
calon = [h.rsplit('/tb/', 1)[0] for h in hit]
calon.sort(key=lambda d: (os.path.exists(d + '/tb/validasi_recursive.py'),
                          os.path.getmtime(d + '/tb/server.py')), reverse=True)
if len(calon) > 1:
    print('PERINGATAN: ada', len(calon), 'salinan kode di input; dipakai yang terbaru.')
    print('  Sebaiknya lepas dataset kode lama dari Input:', calon[1:])
SUMBER = calon[0]
assert os.path.exists(SUMBER + '/data/potongan.jsonl'),     'Folder data/ tidak ikut terunggah; unggah ulang seluruh folder LearningBuddy.'
KERJA = '/kaggle/working/LearningBuddy'
shutil.rmtree(KERJA, ignore_errors=True)
shutil.copytree(SUMBER, KERJA)
os.chdir(KERJA)
sys.path.insert(0, KERJA)
print('kode:', SUMBER, '->', KERJA)

In [ ]:
# Pasang pustaka. Versi dicetak agar dapat dicatat di Bab IV.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt',
                '-r', 'requirements-gpu.txt'], check=True)
import importlib, importlib.metadata as md
importlib.invalidate_caches()
for p in ['torch', 'transformers', 'peft', 'bitsandbytes', 'accelerate',
          'sentence-transformers', 'faiss-cpu', 'langgraph', 'fastapi']:
    try: print(f'{p:22s}', md.version(p))
    except Exception: print(f'{p:22s} TIDAK ADA')
import torch
print('GPU:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Indeks FAISS potongan Recursive (sudah dirapikan) per mapel per kelas, buku Kurikulum
# Merdeka, dengan penyandi pilihan (4.379 potongan, sekitar 5-8 menit).
# Ganti TB_MODEL_EMBEDDING bila hasil tb.uji_embedding memilih penyandi lain.
os.environ.setdefault('TB_MODEL_EMBEDDING', 'Qwen/Qwen3-Embedding-4B')
os.environ.setdefault('TB_DEVICE_CARI', 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda')
print('penyandi', os.environ['TB_MODEL_EMBEDDING'], 'di', os.environ['TB_DEVICE_CARI'])
assert torch.cuda.device_count() > 1, 'Aktifkan Accelerator GPU T4 x2: penyandi 4B butuh GPU kedua'
from tb.pencarian import muat_potongan, bangun_indeks_faiss, folder_indeks, boleh_dicari, MODEL_EMBEDDING
ps = muat_potongan()
meta = os.path.join(folder_indeks(os.environ['TB_MODEL_EMBEDDING']), 'meta.json')
lama = json.load(open(meta)) if os.path.exists(meta) else {}
# dipakai ulang hanya bila indeks tersimpan dibangun dari potongan yang sama
if 'strategi' in lama and lama.get('potongan') == sum(boleh_dicari(p) for p in ps):
    print('indeks sudah ada, dipakai ulang:', lama)
else:
    bangun_indeks_faiss(ps, os.environ['TB_MODEL_EMBEDDING'])
import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Uji pipeline website: 1.014 soal PJOK Merdeka (Validasi_Soal_PJOK), dicari seperti saat
# website berjalan (Recursive, reranker, 3 potongan). Dibandingkan dengan tanpa reranker,
# masing-masing 1 dan 3 potongan. Sekaligus mengkalibrasi ambang pengecekan 1
# (ambang_relevan.json). Sekitar 10-20 menit.
from tb.evaluasi_pencarian import evaluasi
hasil_cari = evaluasi()
os.makedirs('/kaggle/working/hasil', exist_ok=True)
for f in ['evaluasi_pencarian.json', 'Evaluasi_Pencarian.xlsx', 'Validasi_Soal_PJOK_Recursive.xlsx']:
    shutil.copy(f'hasil/{f}', f'/kaggle/working/hasil/{f}')
shutil.copy(os.path.join(folder_indeks(os.environ['TB_MODEL_EMBEDDING']), 'ambang_relevan.json'),
            '/kaggle/working/hasil/ambang_relevan.json')
print('ambang pengecekan 1:', hasil_cari['ambang_relevan']['ambang'], hasil_cari['ambang_relevan']['kinerja'])
import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
shutil.rmtree('/kaggle/working/LearningBuddy', ignore_errors=True)   # output cukup hasil/